In [ ]:
import pyodbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)

In [ ]:
bond = pd.read_csv('C:\\Users\\hermesf\\Projects\\BanksFC\\bond_ts.csv')

In [ ]:
securities = tuple(bond['ISIN'].unique())

In [ ]:
# Data prep
query = f""" 

SELECT business_date, lender_id as entity_id, security_isin, sum(nominal_euro) as lending_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND intragroup = 0
AND s_lender.sector = 'BANK'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin
""" 

df_lend = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 

SELECT business_date, borrower_id as entity_id, security_isin, sum(nominal_euro) as borrowing_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND intragroup = 0
AND s_borrower.sector = 'BANK'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin
""" 

df_borr = pd.read_sql_query(query, cnxn)

In [ ]:
df_lend['business_date'] = pd.to_datetime(df_lend['business_date'])
df_borr['business_date'] = pd.to_datetime(df_borr['business_date'])

In [ ]:
df = df_lend.merge(df_borr, on = ['business_date', 'entity_id', 'security_isin'], how = 'outer')